In [11]:
import pandas as pd
import numpy as np
import importlib
import gc
import sys
import warnings
sys.path.append('../../notebooks/entregable/scripts')  
import dataset
import preprocesamiento
import target
import feature_engineering
importlib.reload(dataset)
importlib.reload(preprocesamiento)
importlib.reload(target)
importlib.reload(feature_engineering)
warnings.filterwarnings("ignore")

In [8]:
df = pd.read_csv("../../data/preprocessed/base.csv", sep=',')
df.shape

(2945818, 13)

In [12]:
#### COMBINATORIA ####
data = dataset.combinatoria_periodo_producto()
data['periodo'] = data['periodo'].dt.year * 100 + data['periodo'].dt.month
data.shape

(44388, 2)

In [13]:
data

,product_id,periodo
0,20524,201701
1,20524,201702
2,20524,201703
3,20524,201704
4,20524,201705
...,...,...
44383,20770,201908
44384,20770,201909
44385,20770,201910
44386,20770,201911


Filtramos los 780 productos

In [32]:
productos_ok = pd.read_csv("../../data/raw/product_id_apredecir201912.csv", sep="\t")
data = data[data['product_id'].isin(productos_ok['product_id'].unique())]
data

,product_id,periodo,cat1,cat2,cat3,brand,sku_size,stock_final,tn,plan_precios_cuidados,cust_request_qty,cust_request_tn
0,20524,201701,HC,VAJILLA,Cristalino,Importado,500,NaN,6.48085,0.0,148.0,6.48085
1,20524,201702,HC,VAJILLA,Cristalino,Importado,500,NaN,3.99755,0.0,121.0,3.99755
2,20524,201703,HC,VAJILLA,Cristalino,Importado,500,NaN,7.14711,0.0,119.0,7.14711
3,20524,201704,HC,VAJILLA,Cristalino,Importado,500,NaN,6.82163,0.0,124.0,6.82163
4,20524,201705,HC,VAJILLA,Cristalino,Importado,500,NaN,9.25949,0.0,161.0,9.25949
...,...,...,...,...,...,...,...,...,...,...,...,...
28075,20127,201908,HC,ROPA LAVADO,Liquido,ROPEX2,3000,NaN,0.00000,NaN,NaN,NaN
28076,20127,201909,HC,ROPA LAVADO,Liquido,ROPEX2,3000,-0.03361,12.80399,0.0,14.0,12.80399
28077,20127,201910,HC,ROPA LAVADO,Liquido,ROPEX2,3000,55.69684,186.81735,0.0,128.0,187.47827
28078,20127,201911,HC,ROPA LAVADO,Liquido,ROPEX2,3000,30.54813,463.80054,0.0,333.0,469.63684


Mergeamos

In [15]:
#### MERGE CON PRODUCTOS ####
productos = pd.read_csv("../../data/raw/tb_productos.csv", sep='\t')
productos = productos.drop_duplicates(subset=['product_id'], keep='first')
data = data.merge(productos, how='left', on="product_id")
del productos

#### MERGE CON STOCKS ####
stocks = pd.read_csv("../../data/raw/tb_stocks.csv", sep='\t')
stocks = stocks.groupby(by=["periodo", "product_id"]).agg({"stock_final": "sum"}).reset_index()
data = data.merge(stocks, how='left', on=['periodo', 'product_id'])
del stocks

#### MERGE CON SELLIN ####
sellin = pd.read_csv("../../data/raw/sell-in.csv", sep='\t')
sellin = sellin.groupby(by=["periodo","product_id"]).agg({"tn":"sum", "plan_precios_cuidados":"sum", "cust_request_qty":"sum", "cust_request_tn":"sum"}).reset_index()
data = data.merge(sellin, how='left', on=['periodo', 'product_id'])
del sellin
gc.collect()

67

Completamos con ceros

In [16]:
#### COMPLETO TN CON CEROS ####
####  ¿cuantos?
print(f"Total de periodos con Nan debido a la combinatoria periodo_x_producto: {data['tn'].isna().sum()}")
#### Lo completo con ceros
data['tn'] = data['tn'].fillna(0)

Total de periodos con Nan debido a la combinatoria periodo_x_producto: 5731


Guardamos

In [18]:
#### GUARDAR DATAFRAME ####
data.to_csv("./datasets/periodo_x_producto.csv", index=False, sep=',', encoding='utf-8')

AutoARIMA

```python
import statsforecast.models as m
print(dir(m))

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, Naive, SeasonalNaive, HoltWinters
from joblib import Parallel, delayed  # Para paralelizar (opcional)

In [24]:
# Creo DF
ts = data[['product_id','periodo', 'tn']].copy()

# Convertir 'periodo' (yyyymm) a datetime
ts['periodo_dt'] = pd.to_datetime(ts['periodo'].astype(str), format='%Y%m')
ts = ts.sort_values(['product_id', 'periodo_dt'])  # Ordenar por producto y fecha

ts.drop(columns=['periodo'], inplace=True)  # Eliminar columna temporal
ts.rename(columns={'tn': 'y', 'periodo_dt':'ds', 'product_id':'unique_id'}, inplace=True)  # Renombrar columna de tn

In [27]:
models = [
    AutoARIMA(season_length=12),
    AutoETS(season_length=12),
    AutoTheta(season_length=12)
]


predictions = {}

for product_id in productos_ok['product_id'].unique():
    df_product = ts[ts['unique_id'] == product_id].copy()
    
    # Añadir columna unique_id (requerida por StatsForecast)
    df_product['unique_id'] = product_id
    
    # Seleccionar columnas necesarias y ordenar por fecha
    df_product = df_product[['unique_id', 'ds', 'y']].sort_values('ds')
    
    if len(df_product) >= 12:
        try:
            sf = StatsForecast(models=models, freq='MS', n_jobs=-1)
            pred = sf.forecast(h=2, df=df_product)
            
            # Obtener la última predicción (h=2)
            pred_feb2020 = pred.sort_values('ds').iloc[[-1]]
            
            # Calcular promedio de modelos
            mean_pred = pred_feb2020[['AutoARIMA', 'AutoETS', 'AutoTheta']].mean(axis=1).iloc[0]
            predictions[product_id] = mean_pred
            
            print(f"Producto {product_id}: Modelo ajustado")
            
        except Exception as e:
            print(f"Error en producto {product_id}: {str(e)}")
            predictions[product_id] = None
    else:
        print(f"Producto {product_id}: Insuficientes datos ({len(df_product)} observaciones)")
        predictions[product_id] = None

# Convertir a DataFrame
df_predictions = pd.DataFrame({
    'product_id': predictions.keys(),
    'prediccion_mes+2': predictions.values()
})

Producto 20001: Modelo ajustado
Producto 20002: Modelo ajustado
Producto 20003: Modelo ajustado
Producto 20004: Modelo ajustado
Producto 20005: Modelo ajustado
Producto 20006: Modelo ajustado
Producto 20007: Modelo ajustado
Producto 20008: Modelo ajustado
Producto 20009: Modelo ajustado
Producto 20010: Modelo ajustado
Producto 20011: Modelo ajustado
Producto 20012: Modelo ajustado
Producto 20013: Modelo ajustado
Producto 20014: Modelo ajustado
Producto 20015: Modelo ajustado
Producto 20016: Modelo ajustado
Producto 20017: Modelo ajustado
Producto 20018: Modelo ajustado
Producto 20019: Modelo ajustado
Producto 20020: Modelo ajustado
Producto 20021: Modelo ajustado
Producto 20022: Modelo ajustado
Producto 20023: Modelo ajustado
Producto 20024: Modelo ajustado
Producto 20025: Modelo ajustado
Producto 20026: Modelo ajustado
Producto 20027: Modelo ajustado
Producto 20028: Modelo ajustado
Producto 20029: Modelo ajustado
Producto 20030: Modelo ajustado
Producto 20031: Modelo ajustado
Producto

In [29]:
df_predictions.to_csv("./outputs/autoarima.csv", index=False, sep=',', encoding='utf-8')

Prophet

In [34]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm

# ---------------------
# 🛠 Preparación inicial
# ---------------------

# Creo DF
df = data[['product_id','periodo', 'tn']].copy()

# Convertir 'periodo' (yyyymm) a datetime
df['periodo_dt'] = pd.to_datetime(df['periodo'].astype(str), format='%Y%m')
df = df.sort_values(['product_id', 'periodo_dt'])  # Ordenar por producto y fecha

df.drop(columns=['periodo'], inplace=True)  # Eliminar columna temporal
df.rename(columns={'tn': 'y', 'periodo_dt':'ds'}, inplace=True)  # Renombrar columna de tn




# Para guardar las predicciones
resultados = []

# ---------------------
#  Loop por producto
# ---------------------
for product_id in productos_ok['product_id'].unique():
    df_prod = df[df['product_id'] == product_id].sort_values('ds')

    if len(df_prod) < 6:
        continue  # opcional: saltear series demasiado cortas

    # Entrenar Prophet
    model = Prophet(yearly_seasonality=True)
    model.fit(df_prod[['ds', 'y']])

    # Predecir mes +2
    ultima_fecha = df_prod['ds'].max()
    future = model.make_future_dataframe(periods=2, freq='MS')
    future = future[future['ds'] > ultima_fecha]  # solo fechas futuras
    pred = model.predict(future)

    # Obtener solo la predicción de mes+2
    pred_mes2 = pred.tail(1)

    resultados.append({
        'product_id': product_id,
        'fecha_predicha': pred_mes2['ds'].values[0],
        'yhat': pred_mes2['yhat'].values[0],
        'yhat_lower': pred_mes2['yhat_lower'].values[0],
        'yhat_upper': pred_mes2['yhat_upper'].values[0]
    })

# ---------------------
# 📊 Resultados finales
# ---------------------
df_predicciones = pd.DataFrame(resultados)
print(df_predicciones.head())


AttributeError: 'Prophet' object has no attribute 'stan_backend'